# 04 — Model Comparison

Final comparison of all four models with real results and selection reasoning.

Results from `scripts/run_quick_train_v2.py` (30% subset) and `scripts/register_best_model.py` (full set).

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Load Experiment Results

In [ ]:
# Load results from the experiment run
with open('../data/processed/model_results.json') as f:
    raw = json.load(f)

# Deduplicate LSTM entries (keep best)
seen = set()
results = []
for r in raw:
    key = r['model']
    if key not in seen:
        seen.add(key)
        results.append(r)

print(f"Loaded {len(results)} model results")
for r in results:
    print(f"  {r['model']:15s}  MAE={r['metrics']['overall']['mae']:6.2f}  R²={r['metrics']['overall']['r2']:.4f}")

## 2. Build Comparison Table

In [ ]:
rows = []
for r in results:
    m = r['metrics']
    for h in m['per_horizon']:
        rows.append({
            'Model': r['model'],
            'Horizon': h['horizon'],
            'MAE': h['mae'],
            'RMSE': h['rmse'],
            'R²': h['r2'],
            'Train Time (s)': r['train_time'],
            'Infer Latency (ms)': r['infer_ms'],
        })
    rows.append({
        'Model': r['model'],
        'Horizon': 'Overall',
        'MAE': m['overall']['mae'],
        'RMSE': m['overall']['rmse'],
        'R²': m['overall']['r2'],
        'Train Time (s)': r['train_time'],
        'Infer Latency (ms)': r['infer_ms'],
    })

df = pd.DataFrame(rows)

# Pivot: models as rows, metrics as columns per horizon
pivot = df.pivot_table(index='Model', columns='Horizon', values=['MAE', 'RMSE', 'R²'], aggfunc='first')
pivot.columns = [f'{v}_{h}' for v, h in pivot.columns]

# Add timing
timing = df[df['Horizon'] == '24h'].set_index('Model')[['Train Time (s)', 'Infer Latency (ms)']]
pivot = pivot.join(timing)

print("FULL COMPARISON TABLE")
print("=" * 80)
pivot.round(4)

## 3. Radar Chart — Multi-Horizon Performance

In [ ]:
import math

models_list = [r['model'] for r in results]
horizons = ['24h', '48h', '72h']
metrics_to_radar = ['MAE', 'RMSE']

fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw=dict(polar=True))
colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']

for ax, metric in zip(axes, metrics_to_radar):
    categories = horizons
    n = len(categories)
    angles = [i / n * 2 * math.pi for i in range(n)]
    angles += angles[:1]

    for i, model_name in enumerate(models_list):
        r_data = [r for r in results if r['model'] == model_name][0]
        values = [r_data['metrics']['per_horizon'][j][metric.lower()] for j in range(3)]
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=model_name, color=colors[i])
        ax.fill(angles, values, alpha=0.1, color=colors[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    ax.set_title(f'{metric} by Horizon', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.suptitle('Multi-Horizon Performance Radar', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

## 4. Complexity vs Performance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for i, r in enumerate(results):
    ax.scatter(r['train_time'], r['metrics']['overall']['mae'],
              s=250, c=colors[i], label=r['model'], zorder=5,
              edgecolors='black', linewidth=0.5)
    ax.annotate(r['model'], (r['train_time'], r['metrics']['overall']['mae']),
               textcoords='offset points', xytext=(10, 5), fontsize=10, fontweight='bold')

ax.set_xlabel('Training Time (s)', fontsize=12)
ax.set_ylabel('Overall MAE (lower = better)', fontsize=12)
ax.set_title('Model Complexity vs Performance', fontsize=14)
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Horizon Degradation Analysis

How much does performance degrade from 24h → 72h?

In [ ]:
degradation = []
for r in results:
    mae_24 = r['metrics']['per_horizon'][0]['mae']
    mae_72 = r['metrics']['per_horizon'][2]['mae']
    pct = (mae_72 - mae_24) / mae_24 * 100
    degradation.append({'Model': r['model'], 'MAE_24h': mae_24, 'MAE_72h': mae_72, 'Degradation %': round(pct, 1)})

deg_df = pd.DataFrame(degradation)
print("Performance degradation from 24h → 72h:")
deg_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(deg_df['Model'], deg_df['Degradation %'], color=colors[:len(deg_df)], edgecolor='white')
ax.set_xlabel('MAE Increase 24h → 72h (%)')
ax.set_title('Horizon Degradation (lower = more stable)')
for bar, val in zip(bars, deg_df['Degradation %']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 6. Selection Reasoning

### Criteria

1. **Performance:** MAE/R² across all horizons
2. **Complexity:** Training time, inference speed, interpretability
3. **Robustness:** Performance degradation from 24h → 72h
4. **Production readiness:** Deployment simplicity

### Baseline Comparison

Ridge serves as the performance floor. Complex models must show meaningful improvement to justify complexity.

### Findings

- **XGBoost** achieves the lowest MAE (21.49) with reasonable training time (16s)
- **RandomForest** performs similarly to XGBoost but is ~9x slower to train
- **Ridge** is competitive — the problem has strong linear relationships in the features
- **LSTM** underperforms — temporal patterns are already captured by lag/rolling features

In [ ]:
# Final ranking
ranking = []
for r in results:
    m = r['metrics']['overall']
    ranking.append({
        'Model': r['model'],
        'Overall MAE': m['mae'],
        'Overall R²': m['r2'],
        'Train Time': f"{r['train_time']:.1f}s",
        'Infer Latency': f"{r['infer_ms']:.3f}ms",
    })

rank_df = pd.DataFrame(ranking).sort_values('Overall MAE')
print("FINAL RANKING (by MAE)")
print("=" * 60)
rank_df

In [ ]:
print("\n" + "=" * 60)
print("RECOMMENDATION")
print("=" * 60)
print("""
XGBoost is the recommended production model.

Reasons:
1. Lowest overall MAE (21.49) — best predictive accuracy
2. Fast training (16s) — practical for retraining
3. Fast inference (0.03ms) — production-ready latency
4. Robust across horizons — consistent degradation pattern
5. Handles missing values natively
6. Feature importance available for interpretability

LSTM underperforms because the feature engineering pipeline
already captures temporal patterns through lag and rolling features,
making the sequential architecture redundant for this dataset.
""")